# មេរៀន 04 - គំរូរចនាប្រើប្រាស់ឧបករណ៍

ក្នុងមេរៀននេះ អ្នកនឹងរៀនអំពីគំរូរចនាប្រើប្រាស់ឧបករណ៍ (**Tool Use**) សម្រាប់ភ្នាក់ងារជំនួយ AI ដោយប្រើ Microsoft Agent Framework (Python)។ យើងគ្របដណ្តប់៖

- ការបញ្ជាក់ឧបករណ៍មុខងារជាមួយ `@tool` decorator និងប៉ារ៉ាម៉ែត្រ typed
- ផ្តល់ schemas ឧបករណ៍ ដើម្បីឱ្យម៉ូឌែលដឹងពីអ្វីដែលឧបករណ៍នីមួយៗធ្វើ
- ការត្រួតពិនិត្យការប្រតិបត្តិឧបករណ៍ជាមួយ `approval_mode`
- ការបង្ហងតម្លៃចេញជា **រចនាសម្ព័ន្ធ** តាមរយៈម៉ូឌែល Pydantic និង `response_format`

ស្ថានភាពគឺជា **ភ្នាក់ងារកក់ដំណើរកម្សាន្ត** ដែលអាចស្វែងរកចំណុចទេសចរណ៍ ពិនិត្យភាពមានស្រាប់ និងយកព័ត៌មានเที่ยวบินវិញបាន។


## ការតាំងខ្នង


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ការប្រើប្រាស់ឧបករណ៍ជាមួយអ្នកតុបតែង @tool

អ្នកតុបតែង `@tool` បម្លែងមុខងារ Python ធម្មតាមួយទៅជាឧបករណ៍ដែលភេជ្ជកម្មអាចហៅបាន។
ចំណុចសំខាន់ៗ៖

- **docstring** គឺក្លាយទៅជាវិពណ៌នាឧបករណ៍ដែលគំរូមើលឃើញ។
- **ការប្រាប់ប្រភេទ** (រួមទាំង `Annotated` ជាមួយនឹងការពិពណ៌នា) កំណត់សំខាន់គំរូឧបករណ៍។
- `approval_mode` គ្រប់គ្រងថាតើអ្នកប្រើត្រូវយល់ព្រមហៅមុខងារពីមុនឬអត់ មុនពេលវាអនុវត្ត។


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## ការបង្កើតភ្នាក់ងារ​ជាមួយ​គ្រឿងចក្រច្រើន

បញ្ជូនឧបករណ៍ទាំងបីទៅកាន់អតិថិជន ដូច្នេះម៉ូដែលអាចហៅប្រើអ្វីក៏បានដែលវាត្រូវការដើម្បីឆ្លើយសំណួររបស់អ្នកប្រើ។ 


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## លទ្ធផលដែលមានរចនាសម្ព័ន្ធជាមួយឧបករណ៍

ដោយកំណត់ `response_format` ទៅជា​ម៉ូដែល Pydantic អ្នកប្រតិបត្តិការត្រូវបានបង្ខំឱ្យត្រឡប់វត្ថុ JSON មួយដែលមានប្រភេទល្អ ជំនួសអត្ថបទទ្រង់ទ្រាយសេរី។ វាអាចប្រើបានពេលកូដខាងក្រោមត្រូវការប្រើលទ្ធផលដោយកម្មវិធី។


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## លំនាំនៃការអនុម័តឧបករណ៍

ប៉ារ៉ាម៉ែត្រ `approval_mode` លើ `@tool` គ្រប់គ្រងថាតើការហៅឧបករណ៍ត្រូវការការអនុម័តពីមនុស្សមុនពេលអនុវត្ត៖

| របៀប | អាកប្បកិរិយា |
|---|---|
| `"never_require"` | ឧបករណ៍ដំណើរការដោយស្វ័យប្រវត្តិ — មិនត្រូវការការបញ្ជាក់ពីអ្នកប្រើទេ។ |
| `"always_require"` | រាល់ការហៅត្រូវតែទទួលការអនុម័តពីអ្នកប្រើមុនពេលវាអនុវត្ត។ |

ប្រើ `"always_require"` សម្រាប់ឧបករណ៍ដែលមានវិជ្ជមានរង (ឧ. ការកក់សំបុត្រហោះហើរ ការទូទាត់កាតឥណទាន) ដូច្នេះមនុស្សនៅក្នុងដំណើរការនោះជានិច្ច។ 


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## សេចក្តីសង្ខេប

នៅក្នុងមេរៀននេះ អ្នកបានរៀនពីរបៀបដូចខាងក្រោម៖

១. **កំណត់ឧបករណ៍** ដោយប្រើអ្នកតុស `@tool` ជាមួយប៉ារ៉ាម៉ែត្រ typed និង docstrings ដែលមានតួនាទីជាស្គីម៉ាឧបករណ៍។
២. **រួមបញ្ចូលឧបករណ៍ច្រើន** ដូច្នេះភ្នាក់ងារអាចហៅវាជាលំដាប់ដើម្បីឆ្លើយសំណួរស្មុគស្មាញ។
៣. **ត្រឡប់លទ្ធផលរចនាសម្ព័ន្ធ** ដោយផ្តល់ម៉ូឌែល Pydantic ជា `response_format`។
៤. **ត្រួតពិនិត្យការអនុម័តឧបករណ៍** ជាមួយ `approval_mode` ដើម្បីរក្សាមនុស្សក្នុងខ្សែសង្វាក់សម្រាប់ប្រតិបត្តិការសង្ខេប។

លំនាំទាំងនេះស្ថិតក្នុងមូលដ្ឋានសម្រាប់សាងសង់ភ្នាក់ងារដែលមានទំនុកចិត្ត និងស្រេចប្រើការបំព្រួញអាចផ្ទេរជាមួយប្រព័ន្ធខាងក្រៅបានយ៉ាងប្រកបដោយសុវត្ថិភាព។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
